# 02 Strict Test-Only Table Model Forecasting

This notebook trains one final model per ticker/model family/horizon under the strict protocol. Hyperparameters are tuned only inside the train block, model families are ranked on validation, and all final artifacts for `03_model_comparison.ipynb` are based on test predictions only. Mature tickers use a rolling maximum of the latest 5 trading years before validation/test fitting; shorter-history tickers use all available history and remain flagged.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "forecasting", cwd.parent, cwd.parent / "forecasting"]:
    if (candidate / "src" / "stock_forecast").exists():
        PROJECT_DIR = candidate
        break
else:
    raise RuntimeError("Cannot locate forecasting project directory with src/stock_forecast")

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
DATA_DIR = ARTIFACT_DIR / "data"
REPORTS_DIR = ARTIFACT_DIR / "reports"
PLOTS_DIR = ARTIFACT_DIR / "plots"
for path in [DATA_DIR, REPORTS_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /Users/romankosenkov/Documents/HSE/year-project/forecasting_stock_prices/forecasting


In [2]:
import importlib.util
import pandas as pd
from IPython.display import display

from stock_forecast.artifacts import load_json, load_table
from stock_forecast.models import build_model
from stock_forecast.strict_protocol import run_strict_per_ticker_protocol

pd.set_option("display.max_columns", 160)


## Notebook Constants

In [3]:
FORCE_RETRAIN = False
PRIMARY_METRIC = "directional_accuracy"
RANDOM_STATE = 42
N_TRIALS = 25
OPTUNA_N_JOBS = 2

STRICT_VALIDATION_ROWS = 126
STRICT_TEST_ROWS = 126
MATURE_MIN_ROWS = 1008
LIMITED_HISTORY_MIN_BLOCK_ROWS = 42
MIN_TRAIN_ROWS = 60
STRICT_MAX_TRAIN_ROWS = 1260
INNER_MAX_FOLDS = 3
INNER_MIN_TRAIN_ROWS = 126
LIMITED_HISTORY_N_TRIALS = 8

LSTM_N_TRIALS = 60
LSTM_LIMITED_HISTORY_N_TRIALS = 20
LSTM_OPTUNA_N_JOBS = 1
LSTM_MAX_EPOCHS = 150
LSTM_PATIENCE = 15
LSTM_DEVICE = "auto"
LSTM_ENSEMBLE_SEEDS = [1, 7, 21, 42, 101]

TRANSACTION_COST_BPS = 10
SLIPPAGE_BPS = 5
LONG_THRESHOLD = 0.0
SIGNAL_ANCHOR = "expanding_median"

HORIZONS = [
    {"name": "week", "horizon": 5},
    {"name": "month", "horizon": 21},
]

MOMENTUM_CANDIDATE_COLUMNS = [
    "ret_lag_1",
    "ret_lag_2",
    "ret_lag_3",
    "ret_lag_5",
    "ret_lag_10",
    "ret_lag_20",
    "rolling_ret_mean_5",
    "rolling_ret_mean_10",
    "rolling_ret_mean_20",
    "rolling_ret_mean_60",
]


def with_tuning(config: dict) -> dict:
    return {**config, "n_trials": N_TRIALS, "optuna_n_jobs": OPTUNA_N_JOBS}


def make_lstm_search_space(horizon: int, limited_history: bool = False) -> dict:
    huber_beta_choices = [0.04, 0.08, 0.12] if horizon >= 21 else [0.02, 0.04, 0.06]
    return {
        "lookback": {"type": "categorical", "choices": [20, 40, 60] if limited_history else [20, 40, 60, 90, 126]},
        "hidden_size": {"type": "categorical", "choices": [16, 32, 64, 96, 128]},
        "num_layers": {"type": "categorical", "choices": [1] if limited_history else [1, 2]},
        "input_projection_size": {"type": "categorical", "choices": [0, 32, 64, 128]},
        "lstm_dropout": {"type": "float", "low": 0.0, "high": 0.35},
        "head_dropout": {"type": "float", "low": 0.15, "high": 0.55},
        "learning_rate": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
        "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
        "batch_size": {"type": "categorical", "choices": [16, 32, 64] if limited_history else [32, 64, 128]},
        "loss": {"type": "categorical", "choices": ["smooth_l1", "mse"]},
        "huber_beta": {"type": "categorical", "choices": huber_beta_choices},
        "feature_clip": {"type": "categorical", "choices": [3.0, 5.0, 8.0]},
        "grad_clip_norm": {"type": "float", "low": 0.5, "high": 2.0},
    }


def make_lstm_config(lstm_feature_cols: list[str], horizon: int) -> dict:
    return {
        "name": "lstm",
        "model_type": "lstm",
        "estimator_factory": build_model,
        "input_mode": "full_frame",
        "feature_cols": lstm_feature_cols,
        "static_params": {
            "max_epochs": LSTM_MAX_EPOCHS,
            "patience": LSTM_PATIENCE,
            "device": LSTM_DEVICE,
        },
        "search_space": make_lstm_search_space(horizon, limited_history=False),
        "limited_history_search_space": make_lstm_search_space(horizon, limited_history=True),
        "post_selection_static_params": {"ensemble_seeds": LSTM_ENSEMBLE_SEEDS},
        "n_trials": LSTM_N_TRIALS,
        "optuna_n_jobs": LSTM_OPTUNA_N_JOBS,
        "limited_history_n_trials": LSTM_LIMITED_HISTORY_N_TRIALS,
        "needs_scaler": False,
    }


def make_model_configs(feature_cols: list[str], lstm_feature_cols: list[str] | None, horizon: int) -> list[dict]:
    momentum_choices = [col for col in MOMENTUM_CANDIDATE_COLUMNS if col in feature_cols]
    configs = [
        {
            "name": "naive_persistence",
            "model_type": "naive_persistence",
            "estimator_factory": build_model,
            "static_params": {},
            "search_space": {},
            "needs_scaler": False,
        },
        with_tuning(
            {
                "name": "momentum",
                "model_type": "momentum",
                "estimator_factory": build_model,
                "static_params": {},
                "search_space": {"column": {"type": "categorical", "choices": momentum_choices}},
                "needs_scaler": False,
            }
        ),
        with_tuning(
            {
                "name": "ridge",
                "model_type": "ridge",
                "estimator_factory": build_model,
                "static_params": {},
                "search_space": {
                    "alpha": {"type": "float", "low": 1e-3, "high": 1e5, "log": True},
                    "solver": {"type": "categorical", "choices": ["auto", "svd", "cholesky", "lsqr"]},
                    "fit_intercept": {"type": "categorical", "choices": [True, False]},
                    "tol": {"type": "float", "low": 1e-6, "high": 1e-2, "log": True},
                },
                "needs_scaler": True,
            }
        ),
        with_tuning(
            {
                "name": "hist_gradient_boosting",
                "model_type": "hist_gradient_boosting",
                "estimator_factory": build_model,
                "static_params": {},
                "search_space": {
                    "max_iter": {"type": "int", "low": 20, "high": 400},
                    "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                    "l2_regularization": {"type": "float", "low": 1e-4, "high": 10.0, "log": True},
                    "max_leaf_nodes": {"type": "int", "low": 3, "high": 31},
                    "max_depth": {"type": "categorical", "choices": [None, 2, 3, 4, 6, 8]},
                    "min_samples_leaf": {"type": "int", "low": 10, "high": 100},
                    "max_bins": {"type": "int", "low": 32, "high": 255},
                },
                "needs_scaler": False,
            }
        ),
    ]

    if importlib.util.find_spec("lightgbm") is not None:
        configs.append(
            with_tuning(
                {
                    "name": "lightgbm",
                    "model_type": "lightgbm",
                    "estimator_factory": build_model,
                    "static_params": {"objective": "regression", "verbosity": -1, "force_col_wise": True, "n_jobs": 1},
                    "search_space": {
                        "n_estimators": {"type": "int", "low": 25, "high": 400},
                        "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                        "num_leaves": {"type": "int", "low": 2, "high": 64},
                        "max_depth": {"type": "categorical", "choices": [-1, 2, 3, 4, 6, 8, 10]},
                        "min_child_samples": {"type": "int", "low": 5, "high": 100},
                        "subsample": {"type": "float", "low": 0.5, "high": 1.0},
                        "subsample_freq": {"type": "int", "low": 1, "high": 10},
                        "colsample_bytree": {"type": "float", "low": 0.5, "high": 1.0},
                        "reg_alpha": {"type": "float", "low": 1e-8, "high": 10.0, "log": True},
                        "reg_lambda": {"type": "float", "low": 1e-3, "high": 100.0, "log": True},
                        "min_split_gain": {"type": "float", "low": 0.0, "high": 1.0},
                    },
                    "needs_scaler": False,
                }
            )
        )

    if importlib.util.find_spec("xgboost") is not None:
        configs.append(
            with_tuning(
                {
                    "name": "xgboost",
                    "model_type": "xgboost",
                    "estimator_factory": build_model,
                    "static_params": {"objective": "reg:squarederror", "verbosity": 0, "n_jobs": 1},
                    "search_space": {
                        "n_estimators": {"type": "int", "low": 25, "high": 400},
                        "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                        "max_depth": {"type": "int", "low": 1, "high": 8},
                        "min_child_weight": {"type": "float", "low": 0.1, "high": 20.0, "log": True},
                        "subsample": {"type": "float", "low": 0.5, "high": 1.0},
                        "colsample_bytree": {"type": "float", "low": 0.5, "high": 1.0},
                        "reg_alpha": {"type": "float", "low": 1e-8, "high": 10.0, "log": True},
                        "reg_lambda": {"type": "float", "low": 1e-3, "high": 100.0, "log": True},
                        "gamma": {"type": "float", "low": 0.0, "high": 5.0},
                    },
                    "needs_scaler": False,
                }
            )
        )

    if importlib.util.find_spec("catboost") is not None:
        configs.append(
            with_tuning(
                {
                    "name": "catboost",
                    "model_type": "catboost",
                    "estimator_factory": build_model,
                    "static_params": {
                        "loss_function": "RMSE",
                        "verbose": False,
                        "allow_writing_files": False,
                        "thread_count": 1,
                        "grow_policy": "Depthwise",
                    },
                    "search_space": {
                        "iterations": {"type": "int", "low": 25, "high": 400},
                        "learning_rate": {"type": "float", "low": 1e-3, "high": 0.1, "log": True},
                        "depth": {"type": "int", "low": 2, "high": 8},
                        "l2_leaf_reg": {"type": "float", "low": 1e-2, "high": 100.0, "log": True},
                        "random_strength": {"type": "float", "low": 0.0, "high": 10.0},
                        "bagging_temperature": {"type": "float", "low": 0.0, "high": 10.0},
                        "border_count": {"type": "int", "low": 32, "high": 254},
                        "min_data_in_leaf": {"type": "int", "low": 1, "high": 50},
                    },
                    "needs_scaler": False,
                }
            )
        )

    if importlib.util.find_spec("torch") is not None and lstm_feature_cols:
        configs.append(make_lstm_config(lstm_feature_cols, horizon))

    return configs

## Load EDA Artifacts

In [4]:
horizon_inputs = []
base_feature_cols = None
TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

for spec in HORIZONS:
    horizon_name = spec["name"]
    horizon_dir = DATA_DIR / "horizons" / horizon_name
    model_dataset_path = horizon_dir / "model_dataset.parquet"
    feature_columns_path = horizon_dir / "feature_columns.json"

    if horizon_name == "week" and not model_dataset_path.exists() and not model_dataset_path.with_suffix(".csv").exists():
        model_dataset_path = DATA_DIR / "model_dataset.parquet"
        feature_columns_path = DATA_DIR / "feature_columns.json"

    if not model_dataset_path.exists() and not model_dataset_path.with_suffix(".csv").exists():
        raise FileNotFoundError(f"Run notebooks/01_eda.ipynb before this notebook; missing {model_dataset_path}")
    if not feature_columns_path.exists():
        raise FileNotFoundError(f"Missing {feature_columns_path}. Run notebooks/01_eda.ipynb first")

    model_df = load_table(model_dataset_path)
    model_df["date"] = pd.to_datetime(model_df["date"])
    feature_payload = load_json(feature_columns_path)
    horizon_feature_cols = feature_payload["feature_columns"]
    target_col = feature_payload["target_column"]
    lstm_feature_cols = None

    if TORCH_AVAILABLE:
        lstm_dir = DATA_DIR / "lstm" / "horizons" / horizon_name
        lstm_model_path = lstm_dir / "model_dataset.parquet"
        lstm_feature_path = lstm_dir / "feature_columns.json"
        if not lstm_model_path.exists() and not lstm_model_path.with_suffix(".csv").exists():
            raise FileNotFoundError(
                f"Missing LSTM artifacts for {horizon_name}: {lstm_model_path}. "
                "Run notebooks/01b_lstm_eda.ipynb before training LSTM."
            )
        if not lstm_feature_path.exists():
            raise FileNotFoundError(f"Missing LSTM feature payload for {horizon_name}: {lstm_feature_path}")
        lstm_payload = load_json(lstm_feature_path)
        if lstm_payload["target_column"] != target_col:
            raise ValueError(f"LSTM target mismatch for {horizon_name}: {lstm_payload['target_column']} != {target_col}")
        model_df = load_table(lstm_model_path)
        model_df["date"] = pd.to_datetime(model_df["date"])
        lstm_feature_cols = lstm_payload["feature_columns"]

    if base_feature_cols is None:
        base_feature_cols = horizon_feature_cols
    elif horizon_feature_cols != base_feature_cols:
        raise ValueError(f"Base feature columns differ for horizon {horizon_name}")

    model_configs = make_model_configs(horizon_feature_cols, lstm_feature_cols, int(feature_payload.get("horizon", spec["horizon"])))
    horizon_inputs.append(
        {
            "horizon_name": feature_payload.get("horizon_name", horizon_name),
            "horizon": int(feature_payload.get("horizon", spec["horizon"])),
            "model_df": model_df,
            "feature_cols": horizon_feature_cols,
            "lstm_feature_cols": lstm_feature_cols,
            "target_col": target_col,
            "model_configs": model_configs,
            "artifact_dir": ARTIFACT_DIR / "horizons" / horizon_name,
        }
    )

    print({
        "horizon": horizon_name,
        "rows": len(model_df),
        "base_features": len(horizon_feature_cols),
        "lstm_features": 0 if lstm_feature_cols is None else len(lstm_feature_cols),
        "target": target_col,
        "models": [model["name"] for model in model_configs],
    })
    display(model_df[["date", "ticker", target_col, *horizon_feature_cols[:6]]].head())

Models: ['naive_persistence', 'momentum', 'ridge', 'hist_gradient_boosting', 'lightgbm', 'xgboost', 'catboost']
{'horizon': 'week', 'rows': 17880, 'features': 127, 'target': 'target_return_5_next_open'}


,date,ticker,target_return_5_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.013316,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.019908,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.005312,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.010582,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,-0.002649,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


{'horizon': 'month', 'rows': 17768, 'features': 127, 'target': 'target_return_21_next_open'}


,date,ticker,target_return_21_next_open,log_close,ret_1,open_close_ret,high_low_range,close_to_high,close_to_low
0,2015-11-09,CBOM,0.027761,1.321756,-0.003992,0.000000,0.000000,0.000000,0.000000
1,2015-11-10,CBOM,0.023842,1.320422,-0.001334,0.004013,0.004005,0.000000,0.004021
2,2015-11-11,CBOM,0.021081,1.319086,-0.001336,0.002677,0.002674,0.000000,0.002681
3,2015-11-12,CBOM,0.021053,1.323088,0.004003,0.000000,0.000000,0.000000,0.000000
4,2015-11-13,CBOM,0.011834,1.329724,0.006636,0.005305,0.300265,-0.227783,0.005319


## Run Strict Protocol

In [5]:
horizon_results = {}

for item in horizon_inputs:
    horizon_name = item["horizon_name"]
    horizon = item["horizon"]
    result = run_strict_per_ticker_protocol(
        model_df=item["model_df"],
        feature_cols=item["feature_cols"],
        target_col=item["target_col"],
        model_configs=item["model_configs"],
        artifact_dir=item["artifact_dir"],
        force_retrain=FORCE_RETRAIN,
        primary_metric=PRIMARY_METRIC,
        random_state=RANDOM_STATE,
        run_metadata={"horizon_name": horizon_name, "horizon": horizon},
        validation_rows=STRICT_VALIDATION_ROWS,
        test_rows=STRICT_TEST_ROWS,
        mature_min_rows=MATURE_MIN_ROWS,
        limited_history_min_block_rows=LIMITED_HISTORY_MIN_BLOCK_ROWS,
        min_train_rows=MIN_TRAIN_ROWS,
        max_train_rows=STRICT_MAX_TRAIN_ROWS,
        inner_max_folds=INNER_MAX_FOLDS,
        inner_min_train_rows=INNER_MIN_TRAIN_ROWS,
        limited_history_n_trials=LIMITED_HISTORY_N_TRIALS,
        transaction_cost_bps=TRANSACTION_COST_BPS,
        slippage_bps=SLIPPAGE_BPS,
        long_threshold=LONG_THRESHOLD,
        signal_anchor=SIGNAL_ANCHOR,
    )
    horizon_results[horizon_name] = result

    print(f"=== Strict protocol: {horizon_name} ({horizon} trading days) ===")
    display(result["outer_splits"])
    display(result["validation_model_ranking"])
    display(result["selected_models_by_ticker"])
    display(result["test_prediction_metrics"])
    display(result["test_signal_metrics"])
    display(result["leakage_audit"])

    failed = result["leakage_audit"][~result["leakage_audit"]["passed"]]
    if not failed.empty:
        raise AssertionError(f"Strict protocol leakage audit failed for {horizon_name}: {failed['check'].tolist()}")


=== Strict protocol: week (5 trading days) ===


,ticker,status,split_quality,limited_history,n_obs,n_train_available,n_train,n_refit_available,n_refit,max_train_rows,n_validation,n_test,train_start,train_end,refit_start,refit_end,validation_start,validation_end,test_start,test_end,train_target_end,refit_target_end,validation_target_end,test_target_end,horizon_name,horizon
0,CBOM,ok,mature,False,2539,2287,1260,2413,1260,1260,126,126,2019-12-02,2024-12-17,2020-06-05,2025-05-23,2024-12-18,2025-05-23,2025-05-26,2025-10-07,2024-12-25,2025-05-31,2025-05-31,2025-10-13,week,5
1,MBNK,ok,limited_history,True,318,190,190,254,254,1260,64,64,2024-09-05,2025-05-21,2024-09-05,2025-07-29,2025-05-22,2025-07-29,2025-07-30,2025-10-07,2025-05-29,2025-08-06,2025-08-06,2025-10-13,week,5
2,SBER,ok,mature,False,4410,4158,1260,4284,1260,1260,126,126,2019-12-02,2024-12-17,2020-06-05,2025-05-23,2024-12-18,2025-05-23,2025-05-26,2025-10-07,2024-12-25,2025-05-31,2025-05-31,2025-10-13,week,5
3,SBERP,ok,mature,False,4410,4158,1260,4284,1260,1260,126,126,2019-12-02,2024-12-17,2020-06-05,2025-05-23,2024-12-18,2025-05-23,2025-05-26,2025-10-07,2024-12-25,2025-05-31,2025-05-31,2025-10-13,week,5
4,SVCB,ok,limited_history,True,417,251,251,334,334,1260,83,83,2024-04-27,2025-04-09,2024-04-27,2025-07-10,2025-04-10,2025-07-10,2025-07-11,2025-10-07,2025-04-15,2025-07-16,2025-07-16,2025-10-13,week,5
5,T,ok,mature,False,1420,1168,1168,1294,1260,1260,126,126,2020-03-13,2024-12-13,2020-04-30,2025-05-21,2024-12-16,2025-05-21,2025-05-22,2025-10-07,2024-12-23,2025-05-29,2025-05-29,2025-10-13,week,5
6,VTBR,ok,mature,False,4366,4114,1260,4240,1260,1260,126,126,2019-11-26,2024-12-17,2020-06-01,2025-05-23,2024-12-18,2025-05-23,2025-05-26,2025-10-07,2024-12-25,2025-05-31,2025-05-31,2025-10-13,week,5


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
3,validation,CBOM,momentum,126,0.050542,0.064387,-0.032487,0.238607,0.135598,0.603175,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,1,True
5,validation,CBOM,ridge,126,0.057839,0.074645,-0.387685,0.016420,0.056540,0.523810,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,2,False
4,validation,CBOM,naive_persistence,126,0.056416,0.070545,-0.239402,0.014613,0.006740,0.500000,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,3,False
2,validation,CBOM,lightgbm,126,0.053503,0.068831,-0.179917,-0.093521,-0.077801,0.452381,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,4,False
1,validation,CBOM,hist_gradient_boosting,126,0.055051,0.070015,-0.220879,-0.089669,-0.068302,0.436508,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,5,False
6,validation,CBOM,xgboost,126,0.051171,0.065275,-0.061157,NaN,NaN,0.420635,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,6,False
0,validation,CBOM,catboost,126,0.052434,0.067718,-0.142078,-0.022527,-0.067333,0.412698,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,7,False
12,validation,MBNK,ridge,64,0.197472,0.248458,-60.245625,0.147357,0.150504,0.640625,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,1,True
11,validation,MBNK,naive_persistence,64,0.029312,0.036138,-0.295647,-0.082903,0.010852,0.546875,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,2,False
13,validation,MBNK,xgboost,64,0.027514,0.032896,-0.073649,NaN,NaN,0.484375,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,3,False


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
3,validation,CBOM,momentum,126,0.050542,0.064387,-0.032487,0.238607,0.135598,0.603175,week,5,mature,False,1254,1260,6,2287,1260,2019-12-02,2024-12-09,1,True
12,validation,MBNK,ridge,64,0.197472,0.248458,-60.245625,0.147357,0.150504,0.640625,week,5,limited_history,True,184,190,6,190,1260,2024-09-05,2025-05-15,1,True
17,validation,SBER,momentum,126,0.027809,0.039988,-0.045619,-0.168581,-0.124199,0.547619,week,5,mature,False,1254,1260,6,4158,1260,2019-12-02,2024-12-09,1,True
23,validation,SBERP,lightgbm,126,0.026205,0.037377,-0.038405,0.010505,0.011066,0.579365,week,5,mature,False,1254,1260,6,4158,1260,2019-12-02,2024-12-09,1,True
32,validation,SVCB,naive_persistence,83,0.034945,0.043822,-0.198469,0.012300,0.005080,0.530120,week,5,limited_history,True,245,251,6,251,1260,2024-04-27,2025-04-03,1,True
41,validation,T,xgboost,126,0.040494,0.054616,-0.022131,NaN,NaN,0.666667,week,5,mature,False,1162,1168,6,1168,1260,2020-03-13,2024-12-05,1,True
46,validation,VTBR,naive_persistence,126,0.050889,0.068006,-0.187993,0.048660,0.093267,0.515873,week,5,mature,False,1254,1260,6,4114,1260,2019-11-26,2024-12-09,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,catboost,126,0.035405,0.045339,-0.177474,0.158048,0.165612,0.444444,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,7,False,0.412698
1,test,CBOM,hist_gradient_boosting,126,0.037630,0.047119,-0.271767,0.086560,0.108110,0.500000,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,5,False,0.436508
2,test,CBOM,lightgbm,126,0.034566,0.043893,-0.103584,0.095166,0.137590,0.515873,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,4,False,0.452381
3,test,CBOM,momentum,126,0.033353,0.042393,-0.029406,-0.104179,-0.196658,0.444444,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,1,True,0.603175
4,test,CBOM,naive_persistence,126,0.035463,0.047183,-0.275220,-0.074744,-0.038584,0.500000,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,3,False,0.500000
5,test,CBOM,ridge,126,0.032458,0.042025,-0.011617,0.180101,0.217746,0.595238,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,2,False,0.523810
6,test,CBOM,xgboost,126,0.032805,0.041804,-0.001034,NaN,NaN,0.507937,week,5,mature,False,1254,1260,6,2413,1260,2020-06-05,2025-05-17,6,False,0.420635
7,test,MBNK,catboost,64,0.027269,0.036585,0.103646,0.565771,0.588233,0.625000,week,5,limited_history,True,248,254,6,254,1260,2024-09-05,2025-07-23,5,False,0.484375
8,test,MBNK,hist_gradient_boosting,64,0.033447,0.041720,-0.165614,-0.183538,-0.163051,0.437500,week,5,limited_history,True,248,254,6,254,1260,2024-09-05,2025-07-23,6,False,0.484375
9,test,MBNK,lightgbm,64,0.029140,0.038651,-0.000412,NaN,NaN,0.625000,week,5,limited_history,True,248,254,6,254,1260,2024-09-05,2025-07-23,4,False,0.484375


...

,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
5,validation,CBOM,ridge,126,0.183240,0.222773,-1.369922,0.477989,0.525369,0.714286,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,1,True
3,validation,CBOM,momentum,126,0.119886,0.145216,-0.007018,0.380199,0.360192,0.674603,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,2,False
6,validation,CBOM,xgboost,126,0.118405,0.148334,-0.050734,-0.585570,-0.444415,0.619048,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,3,False
1,validation,CBOM,hist_gradient_boosting,126,0.120147,0.149107,-0.061706,-0.573815,-0.495051,0.619048,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,4,False
2,validation,CBOM,lightgbm,126,0.121789,0.149149,-0.062307,-0.537117,-0.469035,0.619048,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,5,False
4,validation,CBOM,naive_persistence,126,0.121137,0.145796,-0.015083,0.113085,0.113995,0.563492,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,6,False
0,validation,CBOM,catboost,126,0.127382,0.155058,-0.148142,-0.262248,-0.323231,0.428571,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,7,False
8,validation,MBNK,hist_gradient_boosting,60,0.048051,0.063459,-0.634216,NaN,NaN,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,1,True
12,validation,MBNK,ridge,60,0.048415,0.063871,-0.655535,-0.121510,-0.160711,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,2,False
13,validation,MBNK,xgboost,60,0.049479,0.064837,-0.705989,NaN,NaN,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,3,False


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,validation_directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_train_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected
5,validation,CBOM,ridge,126,0.183240,0.222773,-1.369922,0.477989,0.525369,0.714286,month,21,mature,False,1238,1260,22,2271,1260,2019-11-08,2024-10-24,1,True
8,validation,MBNK,hist_gradient_boosting,60,0.048051,0.063459,-0.634216,NaN,NaN,0.516667,month,21,limited_history,True,160,182,22,182,1260,2024-09-05,2025-04-17,1,True
20,validation,SBER,xgboost,126,0.070686,0.091032,-0.246274,0.045223,0.174429,0.674603,month,21,mature,False,1238,1260,22,4142,1260,2019-11-08,2024-10-24,1,True
24,validation,SBERP,momentum,126,0.067932,0.090109,-0.308032,-0.617028,-0.488336,0.587302,month,21,mature,False,1238,1260,22,4142,1260,2019-11-08,2024-10-24,1,True
28,validation,SVCB,catboost,80,0.049196,0.058185,-0.404646,0.168544,0.071613,0.537500,month,21,limited_history,True,219,241,22,241,1260,2024-04-27,2025-03-04,1,True
37,validation,T,lightgbm,126,0.081024,0.101506,0.066207,0.429886,0.358647,0.658730,month,21,mature,False,1130,1152,22,1152,1260,2020-03-13,2024-10-22,1,True
46,validation,VTBR,naive_persistence,126,0.125237,0.152290,-0.219958,-0.021048,0.009584,0.563492,month,21,mature,False,1238,1260,22,4098,1260,2019-11-01,2024-10-24,1,True


,split_role,ticker,model_name,n_obs,mae,rmse,r2,pearson,spearman,directional_accuracy,horizon_name,horizon,split_quality,limited_history,n_train,n_train_before_purge,n_train_purged,n_refit_available,max_train_rows,train_start,train_end,validation_rank,is_validation_selected,validation_directional_accuracy
0,test,CBOM,catboost,126,0.081123,0.097541,0.045514,0.278584,0.291801,0.650794,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,7,False,0.428571
1,test,CBOM,hist_gradient_boosting,126,0.087954,0.099400,0.008791,0.122738,0.184148,0.444444,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,4,False,0.619048
2,test,CBOM,lightgbm,126,0.085770,0.099409,0.008607,0.126055,0.130726,0.634921,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,5,False,0.619048
3,test,CBOM,momentum,126,0.088952,0.100350,-0.010248,-0.109474,-0.080993,0.230159,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,2,False,0.674603
4,test,CBOM,naive_persistence,126,0.086153,0.100820,-0.019735,0.059501,0.029639,0.531746,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,6,False,0.563492
5,test,CBOM,ridge,126,0.083977,0.104455,-0.094590,0.295857,0.288057,0.619048,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,1,True,0.714286
6,test,CBOM,xgboost,126,0.085955,0.098517,0.026311,0.203773,0.405381,0.761905,month,21,mature,False,1238,1260,22,2397,1260,2020-05-14,2025-04-11,3,False,0.619048
7,test,MBNK,catboost,60,0.086232,0.099695,-0.006644,0.611376,0.607669,0.533333,month,21,limited_history,True,220,242,22,242,1260,2024-09-05,2025-06-25,5,False,0.516667
8,test,MBNK,hist_gradient_boosting,60,0.093132,0.112907,-0.291115,0.692720,0.669023,0.533333,month,21,limited_history,True,220,242,22,242,1260,2024-09-05,2025-06-25,1,True,0.516667
9,test,MBNK,lightgbm,60,0.089818,0.102366,-0.061292,NaN,NaN,0.533333,month,21,limited_history,True,220,242,22,242,1260,2024-09-05,2025-06-25,4,False,0.516667


,cumulative_return,annualized_return,annualized_volatility,periods_per_year,sharpe,sortino,max_drawdown,calmar,turnover,number_of_trades,ticker,model_name,signal_mode,n_rebalances,sample_warning
0,0.037363,0.076123,0.068148,252.0,1.117020,1.495781,-0.104831,0.726151,0.006425,17,CBOM,catboost,overlapping_tranches,126,False
1,-0.001399,-0.002795,0.075787,252.0,-0.036885,-0.069218,-0.134123,-0.020842,0.000378,1,CBOM,hist_gradient_boosting,overlapping_tranches,126,False
2,-0.001399,-0.002795,0.075787,252.0,-0.036885,-0.069218,-0.134123,-0.020842,0.000378,1,CBOM,lightgbm,overlapping_tranches,126,False
3,-0.014546,-0.028881,0.048624,252.0,-0.593971,-1.041034,-0.118213,-0.244313,0.001512,4,CBOM,momentum,overlapping_tranches,126,False
4,-0.012382,-0.024611,0.054982,252.0,-0.447618,-0.507900,-0.071354,-0.344910,0.023054,61,CBOM,naive_persistence,overlapping_tranches,126,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,-0.275729,-0.475432,0.484772,12.0,-0.980733,-1.108389,-0.334896,-1.419641,0.166667,1,VTBR,lightgbm,non_overlapping,6,True
94,-0.194649,-0.351409,0.491674,12.0,-0.714720,-0.627244,-0.260439,-1.349297,0.333333,2,VTBR,momentum,non_overlapping,6,True
95,-0.288406,-0.493633,0.420495,12.0,-1.173933,-0.986322,-0.261547,-1.887358,0.666667,4,VTBR,naive_persistence,non_overlapping,6,True
96,-0.275729,-0.475432,0.484772,12.0,-0.980733,-1.108389,-0.334896,-1.419641,0.166667,1,VTBR,ridge,non_overlapping,6,True


,check,passed,details
0,strict outer splits are available,True,split_rows=7
1,validation predictions are available,True,rows=5390
2,test predictions are available,True,rows=5390
3,outer split dates are chronological,True,bad_rows=0
4,train and refit windows respect max_train_rows,True,"train_over_cap=0, refit_over_cap=0"
5,validation predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
6,test predictions match outer split dates,True,"out_of_window=0, wrong_role=0"
7,final refit target dates end before test starts,True,overlap_rows=0
8,final model payloads exist for test predictions,True,missing_models=0


## Legacy Diagnostics

Legacy walk-forward OOF training is intentionally not part of the primary notebook flow. Final reports must use `strict_protocol/reports/test_predictions.parquet`; if legacy OOF diagnostics are needed later, keep them in a separate notebook/section and never feed them into final comparison plots.